# Wanderbricks Cancellation Predictor
## Databricks ML Overview — Training Session

**Business scenario:** You are a data scientist at Wanderbricks, a fictional travel booking platform. The customer operations team has flagged rising cancellation rates. Your job: build a model that predicts which new bookings are likely to cancel so the team can intervene early with reminders or incentives.

**What we cover:**
1. Explore data in the `samples` catalog
2. Engineer features using Apache Spark
3. Train a classification model with **MLflow autologging**
4. Review the MLflow Experiment UI
5. Evaluate the model
6. Save and reload the model using MLflow
7. Understand how CI/CD fits into the Databricks ML workflow

> **Workspace:** Databricks Individual (free) edition.  
> Steps marked **Paid feature** describe capabilities available in paid or trial workspaces.

In [ ]:
# All libraries are pre-installed in Databricks Runtime — no pip install needed
import mlflow
import mlflow.sklearn
from mlflow import MlflowClient

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, ConfusionMatrixDisplay,
    precision_score, recall_score,
)

print('MLflow version:', mlflow.__version__)

---
## Section 1 — Explore the Data

We read directly from the `samples` catalog — no connection strings, no JDBC setup required.  
Unity Catalog governs access automatically.

**`display()` vs `.show()`:** Always use `display()` in Databricks notebooks. It renders an interactive table with column sorting, filtering, and a built-in chart builder.

In [ ]:
bookings_raw = spark.read.table('samples.wanderbricks.bookings')
display(bookings_raw)

In [ ]:
%sql
SELECT status, COUNT(*) AS booking_count
FROM samples.wanderbricks.bookings
GROUP BY status
ORDER BY booking_count DESC

In [ ]:
%sql
-- Run this first! Verify column names match what the feature engineering code expects.
-- Adjust column names in Section 2 if your schema differs.
DESCRIBE TABLE samples.wanderbricks.bookings

---
## Section 2 — Feature Engineering with Spark

We join four tables with `bookings` as the anchor, derive new columns using `pyspark.sql.functions`, then convert to Pandas for scikit-learn.

| Join | Purpose |
|---|---|
| `bookings → users` | User tenure, age, loyalty tier |
| `bookings → properties` | Property type and average rating |
| `bookings → payments` | Payment method and timing |

> **Why LEFT JOIN?** Not every booking has a completed payment record. Left joins preserve all bookings and let us handle nulls explicitly.

> **Why NOT join `reviews`?** Reviews are written *after* the stay. Using them to predict cancellation would be **data leakage** — the model would train on information it cannot have at prediction time.

In [ ]:
from pyspark.sql import functions as F

bookings   = spark.read.table('samples.wanderbricks.bookings')
users      = spark.read.table('samples.wanderbricks.users')
properties = spark.read.table('samples.wanderbricks.properties')
payments   = spark.read.table('samples.wanderbricks.payments')

print('Bookings  :', bookings.count())
print('Users     :', users.count())
print('Properties:', properties.count())
print('Payments  :', payments.count())

In [ ]:
# --- Join tables (bookings is the anchor) ---
df = (
    bookings
    .join(users.select('user_id', F.col('created_at').alias('user_created_at'), 'country'),
          on='user_id', how='left')
    .join(properties.select('property_id', 'property_type', 'bedrooms', 'base_price'),
          on='property_id', how='left')
    .join(payments.select('booking_id', 'payment_method', 'payment_date'),
          on='booking_id', how='left')
)

# --- Derive features (created_at here is bookings.created_at = booking date) ---
df = (
    df
    # Target variable: 1 = cancelled, 0 = everything else
    .withColumn('is_cancelled',
                F.when(F.col('status') == 'cancelled', 1).otherwise(0))
    # Days between booking date and check-in (longer lead = higher cancel risk)
    .withColumn('lead_time_days',
                F.datediff(F.col('check_in'), F.col('created_at')))
    # Length of the stay in nights
    .withColumn('length_of_stay',
                F.datediff(F.col('check_out'), F.col('check_in')))
    # Month of booking (captures seasonality)
    .withColumn('booking_month',
                F.month(F.col('created_at')))
    # How long the user has been a member at time of booking
    .withColumn('user_tenure_days',
                F.datediff(F.col('created_at'), F.col('user_created_at')))
    # Days between booking and payment (delayed payment is a cancel signal)
    .withColumn('days_to_payment',
                F.datediff(F.col('payment_date'), F.col('created_at')))
)

# --- Select final feature set ---
feature_cols = [
    'is_cancelled',
    'lead_time_days', 'length_of_stay', 'total_amount', 'guests_count', 'booking_month',
    'user_tenure_days', 'country',
    'property_type', 'bedrooms', 'base_price',
    'payment_method', 'days_to_payment',
]

df_features = df.select(*feature_cols)
display(df_features.limit(10))

In [ ]:
# Convert to Pandas for scikit-learn
# Fine for a sample dataset; in production keep everything in Spark or use Spark MLlib
df_pd = df_features.toPandas()

print('Shape:', df_pd.shape)
print('Cancellation rate: {:.1%}'.format(df_pd['is_cancelled'].mean()))
print('\nNull counts:')
print(df_pd.isnull().sum())

In [ ]:
# One-hot encode categoricals
categorical_cols = ['property_type', 'payment_method', 'country']
df_model = pd.get_dummies(df_pd, columns=categorical_cols, drop_first=True)

# Fill nulls from left joins with 0
df_model = df_model.fillna(0)

X = df_model.drop('is_cancelled', axis=1)
y = df_model['is_cancelled']

# stratify=y ensures both splits have the same cancellation rate
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Training samples:', len(X_train), ' | Test samples:', len(X_test))
print('Cancellation rate in train: {:.1%}'.format(y_train.mean()))
print('Number of features:', X.shape[1])

---
## Section 3 — Train with MLflow Autologging

`mlflow.sklearn.autolog()` is the single most impactful Databricks ML feature for beginners.  
**One line** automatically captures:
- All model hyperparameters (`n_estimators`, `max_depth`, …)
- Training metrics (accuracy, F1, precision, recall)
- The serialised model artifact
- Feature importance plots

Without autolog, you would write 20+ `mlflow.log_param()` calls by hand.

---

**Why `class_weight='balanced'`?**  
Cancellations are a minority class — typically 10–20% of all bookings.  
Without this, the model learns to always predict *'not cancelled'* and still achieves 80%+ accuracy — without actually learning anything useful.

In [ ]:
# Group all experiment runs under one named experiment
mlflow.set_experiment('/Users/kar.lok.soo@accenture.com/wanderbricks_cancellation')

# One line replaces 20+ mlflow.log_param() calls
mlflow.sklearn.autolog(log_input_examples=True, log_model_signatures=True)

with mlflow.start_run(run_name='rf_baseline_v1') as run:
    rf = RandomForestClassifier(
        n_estimators=100,
        max_depth=8,
        class_weight='balanced',  # critical for imbalanced cancellation data
        random_state=42,
        n_jobs=-1
    )
    rf.fit(X_train, y_train)

    # Test-set metrics
    y_prob      = rf.predict_proba(X_test)[:, 1]
    y_pred_test = rf.predict(X_test)
    test_auc       = roc_auc_score(y_test, y_prob)
    test_precision = precision_score(y_test, y_pred_test)
    test_recall    = recall_score(y_test, y_pred_test)

    # Training-set metrics (detect overfitting when compared to test)
    y_pred_train    = rf.predict(X_train)
    train_precision = precision_score(y_train, y_pred_train)
    train_recall    = recall_score(y_train, y_pred_train)

    mlflow.log_metric('test_roc_auc',     test_auc)
    mlflow.log_metric('test_precision',   test_precision)
    mlflow.log_metric('test_recall',      test_recall)
    mlflow.log_metric('train_precision',  train_precision)
    mlflow.log_metric('train_recall',     train_recall)

    run_id = run.info.run_id

print(f'Run ID         : {run_id}')
print(f'Test  AUC      : {test_auc:.4f}')
print(f'Test  Precision: {test_precision:.4f}  |  Test  Recall: {test_recall:.4f}')
print(f'Train Precision: {train_precision:.4f}  |  Train Recall: {train_recall:.4f}')

---
## Section 4 — MLflow Experiment UI Walkthrough

> **Live demo — no new code in this section.**

Navigate to the **Experiments** tab in the left sidebar → open **wanderbricks_cancellation**.

| Tab | What you see |
|---|---|
| **Parameters** | `n_estimators`, `max_depth`, `class_weight` — auto-captured |
| **Metrics** | accuracy, F1, precision, recall, `test_roc_auc` |
| **Artifacts** | Serialised model, `MLmodel` manifest, feature importance plot |

**Key insight:** This is your audit trail.  
Six months from now you will know exactly what data, what code, and what parameters produced this model — without maintaining a spreadsheet.

**Try this:** Run the training cell again with `n_estimators=50`, then click **Compare runs** in the UI to see the difference side by side.

---
## Section 5 — Evaluate the Model

Three visualisations — no more:

1. **AUC score** — primary metric for imbalanced classification
2. **Confusion matrix** — true/false positives and negatives at a glance
3. **Feature importance** — which signals actually drive the prediction

**Why AUC and not accuracy?**  
If 15% of bookings cancel, a model that always predicts *'no cancel'* achieves 85% accuracy — and is completely useless.  
AUC measures whether the model consistently ranks cancellations *above* non-cancellations, regardless of threshold.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix
ConfusionMatrixDisplay.from_estimator(
    rf, X_test, y_test, ax=axes[0], colorbar=False
)
axes[0].set_title('Confusion Matrix')

# Feature importance — top 10
importance = pd.Series(rf.feature_importances_, index=X.columns)
importance.nlargest(10).sort_values().plot(kind='barh', ax=axes[1], color='steelblue')
axes[1].set_title('Top 10 Feature Importances')
axes[1].set_xlabel('Importance')

plt.tight_layout()
plt.show()

print('Test AUC: {:.3f}'.format(test_auc))

---
## Section 6 — Save and Load the Model with MLflow

On the Individual (free) edition, we load the model directly from the MLflow run URI — no separate model registry step required.

**What this looks like in a paid workspace:**
```python
mlflow.set_registry_uri('databricks-uc')
mlflow.register_model(model_uri, 'main.default.cancellation_predictor')

# Set a stable alias
client.set_registered_model_alias('main.default.cancellation_predictor', 'champion', version)

# Inference pipelines always reference the alias, not the version number
mlflow.pyfunc.load_model('models:/main.default.cancellation_predictor@champion')
```
When you promote a new model, you move the `@champion` alias. The inference pipeline code never changes.

In [ ]:
# Load the model directly from the MLflow run artifact (works on free edition)
model_uri = 'runs:/' + run_id + '/model'
loaded_model = mlflow.sklearn.load_model(model_uri)

# Score 5 test samples — proves the round-trip works
sample_input  = X_test.head(5)
predictions   = loaded_model.predict(sample_input)
probabilities = loaded_model.predict_proba(sample_input)[:, 1]

result = sample_input[['lead_time_days', 'total_amount']].copy()
result['predicted_cancelled']      = predictions
result['cancellation_probability'] = probabilities.round(3)

display(result)

---
## Section 7 — CI/CD in Databricks: Conceptual Overview

On the free edition, full CI/CD cannot be demonstrated live — but the pattern is straightforward:

```
Developer pushes notebook/code to GitHub
          |
          v
GitHub Actions triggers on PR or merge to main
          |
          v
databricks bundle deploy  (deploys code to staging workspace)
          |
          v
Databricks Workflow runs automated tests
(unit tests + model quality validation)
          |
          v
On success: model alias promoted to @champion in prod workspace
          |
          v
Inference pipeline always loads @champion -- no code changes needed
```

**Key tools:**

| Tool | Role |
|---|---|
| **Databricks Asset Bundles (DABs)** | Infrastructure-as-code — define notebooks, jobs, clusters in `databricks.yml` |
| **Databricks CLI** | Runs `databricks bundle deploy` from GitHub Actions |
| **MLflow Model Registry** | Model versioning, aliases, and lineage (Unity Catalog — paid feature) |
| **Databricks Workflows** | Orchestration for scheduled and event-triggered jobs |

> You can version this notebook in Git right now — connect via **Repos** in the workspace sidebar and commit/push to GitHub.  
> The multi-environment deployment step requires a trial or paid workspace.

---
## Summary

| Step | What we used | Why it matters |
|---|---|---|
| Data access | `spark.read.table()` + `samples` catalog | No connection strings; governed access |
| Feature engineering | PySpark `withColumn()`, `datediff()` | Scalable; same code works on billions of rows |
| Model training | `RandomForestClassifier` + `mlflow.sklearn.autolog()` | Full experiment tracking in one line |
| Multi-model benchmark | LR, GBM, XGBoost, RandomForest | Compare algorithms on equal footing |
| Feature importance | `feature_importances_`, `coef_` | Understand *why* the model predicts what it does |
| Experiment tracking | MLflow Experiments UI | Reproducible, auditable, comparable runs |
| Model Registry | `mlflow.register_model()` + aliases | Stable `@champion` pointer — no version hardcoding |
| Batch serving | `mlflow.sklearn.load_model('models:/...@champion')` | Load once, score any DataFrame |
| Real-time serving | Databricks Serving endpoint (paid feature) | REST API with auto-scaling; alias-driven promotion |
| CI/CD | Databricks Asset Bundles + GitHub Actions | Automated, governed ML deployment |

**Try next:**
- In the **Models tab**: click `wanderbricks_cancellation_predictor` → compare versions → manually re-assign `@champion` to a different version
- In the **Experiments tab**: select all four runs → click **Compare** → chart `test_roc_auc` across runs
- Tune XGBoost hyperparameters and re-register — watch the version list grow
- Explore **Databricks AutoML** for automated feature engineering and model selection

In [ ]:
# ── How to promote a new champion (e.g. after next retraining cycle) ─────────
#
# 1. Train new model, get its run_id
# 2. Register it:
#       new_mv = mlflow.register_model('runs:/<new_run_id>/model', REGISTRY_NAME)
# 3. Evaluate, then promote:
#       client.set_registered_model_alias(REGISTRY_NAME, 'champion', new_mv.version)
# 4. The serving endpoint (if configured) immediately picks up the new version.
#    No code changes needed anywhere in the inference pipeline.
#
# To inspect current registry state:
print(f'All registered versions of {REGISTRY_NAME}:\n')
for mv in client.search_model_versions(f"name='{REGISTRY_NAME}'"):
    tags    = mv.tags
    aliases = ', '.join(mv.aliases) if mv.aliases else '—'
    print(f'  v{mv.version:<4}  {tags.get("model_type",""):<24}  AUC={tags.get("test_roc_auc","?")}  aliases=[{aliases}]  run={mv.run_id[:8]}...')

In [ ]:
# ── Option A: Load @champion and batch-score ─────────────────────────────────
champion_uri   = f'models:/{REGISTRY_NAME}@champion'
champion_model = mlflow.sklearn.load_model(champion_uri)

print(f'Loaded champion: {REGISTRY_NAME}@champion  (version {champion_version})')
print(f'Model type: {type(champion_model).__name__}\n')

# Score the full test set
y_prob_champion = champion_model.predict_proba(X_test)[:, 1]
y_pred_champion = champion_model.predict(X_test)

final_auc = roc_auc_score(y_test, y_prob_champion)
print(f'Champion Test AUC: {final_auc:.4f}')

# Show top 10 highest-risk bookings in the test set
risk_df = X_test[['lead_time_days', 'total_amount', 'length_of_stay']].copy()
risk_df['cancel_probability'] = y_prob_champion.round(4)
risk_df['predicted_cancel']   = y_pred_champion
risk_df['actual_cancel']      = y_test.values

top_risk = risk_df.sort_values('cancel_probability', ascending=False).head(10)
print('\nTop 10 highest-risk bookings (champion model):')
display(top_risk)

---
## Section 11 — Serve the Champion Model

### Option A — Batch scoring (works on free edition)

Load the `@champion` alias and score a DataFrame directly in the notebook.  
This is the pattern for nightly batch jobs: load once, score millions of rows.

### Option B — Real-time REST endpoint (paid / Trial workspace only)

```
Serving tab (left sidebar) → Create serving endpoint
  Model: wanderbricks_cancellation_predictor
  Served entity: @champion   ← always follows the alias
  Compute: Small (for demo)
```

The endpoint auto-scales and exposes a `/invocations` REST API:
```bash
curl -X POST https://<workspace-url>/serving-endpoints/cancellation-predictor/invocations \
  -H 'Authorization: Bearer <token>' \
  -H 'Content-Type: application/json' \
  -d '{"dataframe_records": [{"lead_time_days": 45, "total_amount": 320.0, ...}]}'
```

**Why alias-based serving?**  
When you promote a new champion (e.g. next month's retrained XGBoost), you move the `@champion` alias.  
The endpoint immediately serves the new version — zero config change, zero downtime.

In [ ]:
client         = MlflowClient()
REGISTRY_NAME  = 'wanderbricks_cancellation_predictor'

# ── Register every model run in the registry ──────────────────────────────────
print(f'Registering all models as versions of: {REGISTRY_NAME}\n')
registered = []

for result in model_results:
    model_uri = f'runs:/{result["run_id"]}/model'
    mv = mlflow.register_model(model_uri=model_uri, name=REGISTRY_NAME)
    # Tag each version with its name and AUC for easy filtering in the UI
    client.set_model_version_tag(REGISTRY_NAME, mv.version, 'model_type',  result['model_name'])
    client.set_model_version_tag(REGISTRY_NAME, mv.version, 'test_roc_auc', f'{result["test_auc"]:.4f}')
    registered.append({'model_name': result['model_name'], 'version': mv.version,
                        'test_auc': result['test_auc'], 'run_id': result['run_id']})
    print(f'  Registered {result["model_name"]:<24} → version {mv.version}  (AUC={result["test_auc"]:.4f})')

# ── Identify the best version by Test AUC ─────────────────────────────────────
best_entry = max(registered, key=lambda r: r['test_auc'])
print(f'\nBest model: {best_entry["model_name"]}  v{best_entry["version"]}  (AUC={best_entry["test_auc"]:.4f})')

# ── Assign the @champion alias to the best version ────────────────────────────
client.set_registered_model_alias(
    name    = REGISTRY_NAME,
    alias   = 'champion',
    version = best_entry['version'],
)
print(f'\n@champion alias → {best_entry["model_name"]} (version {best_entry["version"]})')
print(f'\nNavigate to: Models tab (left sidebar) → {REGISTRY_NAME}')
print('You will see all versions with tags, AUC, and the @champion alias.')

champion_version = best_entry['version']

---
## Section 10 — Register All Models in the MLflow Model Registry

The **Model Registry** is the handoff point between experimentation and production.  
Every model is registered as a new version under the same name — so the **Models tab** shows all versions, their metrics, and their current alias in one place.

**Workflow:**
```
Train (experiment run)  →  Register (Model Registry)  →  Assign alias  →  Inference pipeline loads @champion
```

**Aliases vs version numbers:**
- Version numbers auto-increment every time you register. Never hardcode them in inference code.
- Aliases (`@champion`, `@challenger`) are stable pointers you move between versions. Promote a new model by moving the alias — no pipeline code changes needed.

> On the **free Databricks edition**: models register to the workspace-level registry (visible in the **Models** tab in the left sidebar).  
> On **paid / Trial with Unity Catalog**: prefix the model name with a three-level namespace, e.g. `main.ml.cancellation_predictor`, and set `mlflow.set_registry_uri('databricks-uc')` first.

In [ ]:
# ── Side-by-side feature importance for all models ───────────────────────────
models_with_importance = [r for r in model_results if r['importance'] is not None]

cols = 2
rows = (len(models_with_importance) + 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(16, 5 * rows))
axes = axes.flatten()

colors = ['steelblue', 'darkorange', 'forestgreen', 'mediumpurple']

for i, result in enumerate(models_with_importance):
    imp   = result['importance']
    top10 = imp.nlargest(10).sort_values()
    top10.plot(kind='barh', ax=axes[i], color=colors[i % len(colors)])
    axes[i].set_title(f'{result["model_name"]}\n(AUC = {result["test_auc"]:.4f})', fontsize=11)
    axes[i].set_xlabel('Importance Score')

# Hide any unused subplots
for j in range(len(models_with_importance), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Feature Importance — Top 10 per Model', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

# ── Aggregate: which features appear in every model's top 10? ────────────────
top10_sets = [set(r['importance'].nlargest(10).index) for r in models_with_importance]
universal  = set.intersection(*top10_sets) if top10_sets else set()
print('\nFeatures in the top-10 of ALL models (universal signals):')
for f in sorted(universal):
    print(' ', f)

---
## Section 9 — Feature Importance Comparison

Plotting all four models side-by-side reveals which signals are universally important vs model-specific.

- **Tree models** (RF, GBM, XGB): importance = mean decrease in impurity across all splits on that feature
- **Logistic Regression**: importance = absolute value of the coefficient (after standardisation — units are comparable)

A feature that ranks highly in *all* models is a robust, trustworthy signal.  
A feature that only matters to one model may be capturing noise or a model-specific artifact.

In [ ]:
# ── Model Comparison Table ────────────────────────────────────────────────────
metric_cols = ['model_name', 'test_auc', 'test_precision', 'test_recall',
               'train_precision', 'train_recall', 'run_id']

comparison_df = (
    pd.DataFrame(model_results)[metric_cols]
    .sort_values('test_auc', ascending=False)
    .reset_index(drop=True)
)
comparison_df.index += 1  # 1-based rank
comparison_df.columns = ['Model', 'Test AUC', 'Test Precision', 'Test Recall',
                          'Train Precision', 'Train Recall', 'Run ID']
for col in ['Test AUC', 'Test Precision', 'Test Recall', 'Train Precision', 'Train Recall']:
    comparison_df[col] = comparison_df[col].round(4)

print('=== Model Leaderboard ===')
print(comparison_df.to_string())

# Highlight overfitting: large gap between train and test precision/recall
print('\n=== Overfitting Check (Train − Test gap) ===')
for r in model_results:
    prec_gap = r['train_precision'] - r['test_precision']
    rec_gap  = r['train_recall']    - r['test_recall']
    flag = '  ⚠ possible overfit' if prec_gap > 0.15 or rec_gap > 0.15 else ''
    print(f"  {r['model_name']:<24}  precision gap={prec_gap:+.4f}  recall gap={rec_gap:+.4f}{flag}")

best = max(model_results, key=lambda r: r['test_auc'])
print(f'\nBest model: {best["model_name"]}  (AUC={best["test_auc"]:.4f})')

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_sample_weight
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

mlflow.set_experiment('/Users/kar.lok.soo@accenture.com/wanderbricks_cancellation')
mlflow.sklearn.autolog(log_input_examples=True, log_model_signatures=True)

model_results = []

# ── Include the Random Forest trained in Section 3 ──────────────────────────
rf_pred_test    = rf.predict(X_test)
rf_pred_train   = rf.predict(X_train)
rf_auc          = roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1])
rf_test_prec    = precision_score(y_test,  rf_pred_test)
rf_test_rec     = recall_score(y_test,     rf_pred_test)
rf_train_prec   = precision_score(y_train, rf_pred_train)
rf_train_rec    = recall_score(y_train,    rf_pred_train)
rf_importance   = pd.Series(rf.feature_importances_, index=X.columns)

model_results.append({
    'model_name':      'RandomForest',
    'run_id':          run_id,
    'test_auc':        rf_auc,
    'test_precision':  rf_test_prec,
    'test_recall':     rf_test_rec,
    'train_precision': rf_train_prec,
    'train_recall':    rf_train_rec,
    'importance':      rf_importance,
    'model':           rf,
})
print(f'  [RandomForest]  AUC={rf_auc:.4f}  test_prec={rf_test_prec:.4f}  test_rec={rf_test_rec:.4f}  (from Section 3)')


def get_importance(model, feature_names):
    """Extract feature importances from any supported sklearn-compatible model."""
    estimator = model.named_steps.get('clf') if hasattr(model, 'named_steps') else model
    if hasattr(estimator, 'feature_importances_'):
        return pd.Series(estimator.feature_importances_, index=feature_names)
    elif hasattr(estimator, 'coef_'):
        return pd.Series(np.abs(estimator.coef_[0]), index=feature_names)
    return None


def train_and_log(model, model_name, fit_params=None):
    """Train a model, log all metrics to MLflow, save feature importance artifact."""
    fit_params = fit_params or {}
    with mlflow.start_run(run_name=model_name) as run:
        model.fit(X_train, y_train, **fit_params)

        # ── Test-set metrics ──────────────────────────────────────────────────
        y_prob      = model.predict_proba(X_test)[:, 1]
        y_pred_test = model.predict(X_test)
        auc            = roc_auc_score(y_test, y_prob)
        test_precision = precision_score(y_test, y_pred_test)
        test_recall    = recall_score(y_test,    y_pred_test)

        # ── Training-set metrics (compare to test to spot overfitting) ────────
        y_pred_train    = model.predict(X_train)
        train_precision = precision_score(y_train, y_pred_train)
        train_recall    = recall_score(y_train,    y_pred_train)

        mlflow.log_param('model_type',       model_name)
        mlflow.log_metric('test_roc_auc',    auc)
        mlflow.log_metric('test_precision',  test_precision)
        mlflow.log_metric('test_recall',     test_recall)
        mlflow.log_metric('train_precision', train_precision)
        mlflow.log_metric('train_recall',    train_recall)

        importance = get_importance(model, X_train.columns.tolist())
        if importance is not None:
            fig, ax = plt.subplots(figsize=(8, 5))
            importance.nlargest(10).sort_values().plot(kind='barh', ax=ax, color='steelblue')
            ax.set_title(f'{model_name} — Top 10 Feature Importances')
            ax.set_xlabel('Importance Score')
            plt.tight_layout()
            mlflow.log_figure(fig, 'feature_importance.png')
            plt.close(fig)

        stored_run_id = run.info.run_id

    print(f'  [{model_name:<22}]  AUC={auc:.4f}  '
          f'test_prec={test_precision:.4f}  test_rec={test_recall:.4f}  '
          f'train_prec={train_precision:.4f}  train_rec={train_recall:.4f}')
    return {
        'model_name':      model_name,
        'run_id':          stored_run_id,
        'test_auc':        auc,
        'test_precision':  test_precision,
        'test_recall':     test_recall,
        'train_precision': train_precision,
        'train_recall':    train_recall,
        'importance':      importance,
        'model':           model,
    }


print('\nTraining additional models...')

# ── Logistic Regression ───────────────────────────────────────────────────────
lr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)),
])
model_results.append(train_and_log(lr_pipeline, 'LogisticRegression'))

# ── Gradient Boosting ─────────────────────────────────────────────────────────
gbm = GradientBoostingClassifier(n_estimators=200, max_depth=5, learning_rate=0.05,
                                  subsample=0.8, random_state=42)
sw  = compute_sample_weight('balanced', y_train)
model_results.append(train_and_log(gbm, 'GradientBoosting',
                                   fit_params={'sample_weight': sw}))

# ── XGBoost ───────────────────────────────────────────────────────────────────
neg_pos_ratio = (y_train == 0).sum() / (y_train == 1).sum()
xgb_clf = xgb.XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.05,
                              subsample=0.8, colsample_bytree=0.8,
                              scale_pos_weight=neg_pos_ratio,
                              eval_metric='auc', random_state=42,
                              verbosity=0, use_label_encoder=False)
model_results.append(train_and_log(xgb_clf, 'XGBoost'))

print('\nDone. All runs logged to experiment: wanderbricks_cancellation')

---
## Section 8 — Train Multiple Models & Compare

We benchmark **four classifiers** on the same feature set and split.  
Each model gets its own MLflow run under the same experiment — navigate to the **Experiments** tab → `wanderbricks_cancellation` → **Compare runs** to see all results side by side.

| Model | Key strength | Class imbalance handling |
|---|---|---|
| **Logistic Regression** | Fast linear baseline; coefficients give direction and magnitude | `class_weight='balanced'` |
| **Gradient Boosting** | Sequential ensemble; strong on tabular data | `sample_weight` computed from training labels |
| **XGBoost** | Regularised gradient boosting; GPU-ready; best single model on most tabular benchmarks | `scale_pos_weight` = negative / positive ratio |
| **Random Forest** | Already trained in Section 3; included via stored `run_id` for comparison | `class_weight='balanced'` |

> **Tip:** After all runs complete, click the **chart icon** in the Experiments UI to see AUC on a bar chart across all four runs.